
# TrustCV Deep Learning Showcase (No Indexable* helpers)
This notebook is the **same story and same experiments** as the previous “showcase”, but it **removes** the `IndexableDict / IndexableList / IndexableTFDataset` helpers.

We rely on what you described in your library:

- **`UniversalCVRunner → SklearnAdapter.create_data_splits(...)`** recursively slices **dicts** and **lists/tuples** using the fold indices (and uses `.iloc` for pandas objects).
- **`KerasSkWrap` / `KerasRegressorWrap`** accept **dict/list inputs** and pass them directly into Keras.
- **Dict outputs** are handled via `output_key` / `output_index` selection (no extra wrapping needed), as long as each value supports `v[idx]`.

## How to read the results (the “numbers that matter”)
Across all experiments, interpret the outputs using three kinds of evidence:

1. **Patient overlap (leakage sanity check)**  
   In group-safe CV, the number of patient IDs shared between train/test in each fold should be **0**.

2. **Per-fold metrics stability**  
   Look at the spread across folds (std). Big std often means the model depends on a subset of patients or labels are imbalanced.

3. **Prevalence per fold** (especially for multilabel)  
   If the label prevalence differs a lot across folds, macro metrics will swing more than micro metrics.

---



## Feature glossary (units, typical ranges, clinical interpretation)

| Feature | Unit | Typical range (adult) | One-line clinical interpretation |
|---|---:|---:|---|
| **age** | years | ~18–90+ | Older age generally increases cardiovascular risk (events, HF, stroke). |
| **bmi** | kg/m² | ~18–40+ | Higher BMI is associated with cardiometabolic risk (HTN, diabetes, dyslipidemia). |
| **sbp** | mmHg | ~90–180+ | Elevated systolic BP increases risk of MI, stroke, HF, and kidney disease. |
| **ef** | % | ~35–70 | Left ventricular ejection fraction; lower EF suggests impaired pump function and higher HF/event risk. |
| **lvmass** (LV mass index) | g/m² | ~50–150 | Higher LV mass indicates remodeling/LVH, often from chronic pressure overload (e.g., HTN). |
| **af** | 0/1 | 0 or 1 | Atrial fibrillation present; associated with stroke risk and adverse cardiac outcomes. |
| **lvh** | 0/1 | 0 or 1 | Left ventricular hypertrophy flag; marker of structural heart disease and elevated risk. |
| **cond** | 0/1 | 0 or 1 | Conduction abnormality flag (e.g., bundle branch block); may reflect structural/electrical disease. |
| **mace_1y** | 0/1 | 0 or 1 | Major adverse cardiovascular event within 1 year (binary outcome to classify). |
| **ef_drop** | %-points | ~0–20 | Future EF decline magnitude; larger values indicate worsening LV function (regression target). |

**Notes (for this synthetic demo):**
- Ranges are approximate and intended for intuition; real cohorts vary by inclusion criteria and measurement modality.
- `af`, `lvh`, `cond` can co-occur (multi-label setting).
- `ef_drop` is expressed as **percentage-point drop** in EF (e.g., 55% → 48% is a 7-point drop).


In [ ]:

# --- Colab install (uncomment in Colab) ---
# !pip -q install trustcv==1.0.5 tensorflow scikit-learn pandas matplotlib

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    mean_squared_error, r2_score
)

import trustcv
from trustcv import TrustCVValidator, UniversalCVRunner
from trustcv.frameworks.tensorflow_sklearn import KerasSkWrap, KerasRegressorWrap
from trustcv.splitters.grouped import StratifiedGroupKFold

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

print("trustcv version:", trustcv.__version__)
print("tf version:", tf.__version__)
print("KerasSkWrap:", KerasSkWrap)



## Step 1 — Simulate a small cardiology-like dataset (patient IDs + repeated visits)

**Scenario:** Each patient has multiple visits. If you split by sample (row) instead of by patient, the model can “see” the same patient in train and test.  
This is the classic **patient leakage** problem.

We generate:
- `X_tab`: clinical features (age, BMI, SBP)
- `X_img`: imaging biomarkers (EF, LV mass index)
- `X_dict`: multi-input dict (`{"clinical": ..., "imaging": ...}`)
- `X_list`: multi-input list (`[X_tab, X_img]`)
- `y_main`: binary MACE (1-year)
- `y_multi`: multilabel flags (AF/LVH/Conduction), C=3
- `y_future_ef_drop`: regression target
- `y_multiout`: dict outputs for multi-head model (`{"mace":..., "ef_drop":...}`)

**Judgment numbers:** confirm
- `N_patients` and `visits_per_patient` are correct
- label prevalence is reasonable (not 0% or 100%)


In [ ]:

# -------------------------
# Data generation
# -------------------------
N_patients = 200
visits_per_patient = 4
N = N_patients * visits_per_patient

patient_ids = np.repeat(np.arange(N_patients), visits_per_patient)

# Clinical features
age = np.random.normal(65, 10, N)
bmi = np.random.normal(27, 4, N)
sbp = np.random.normal(135, 15, N)

# Imaging biomarkers
ef = np.random.normal(55, 8, N)
lvmass = np.random.normal(90, 20, N)

# Multi-label ECG/echo flags (C=3)
af_flag   = (np.random.rand(N) < 0.25).astype(int)
lvh_flag  = (np.random.rand(N) < 0.30).astype(int)
cond_flag = (np.random.rand(N) < 0.20).astype(int)
y_multi = np.stack([af_flag, lvh_flag, cond_flag], axis=1).astype("int32")

# Binary outcome: 1y MACE risk (depends on age, EF, flags + latent patient effect)
patient_risk = np.random.normal(0, 0.7, N_patients)  # latent patient effect
logit = (
    0.03*(age-60)
    -0.05*(ef-55)
    +0.8*af_flag +0.5*lvh_flag +0.6*cond_flag
    + patient_risk[patient_ids]
)
prob = 1/(1+np.exp(-logit))
y_main = (np.random.rand(N) < prob).astype("int32")

# Regression outcome: future EF drop
y_future_ef_drop = (
    np.maximum(0, 10 - 0.10*(ef-55)) + 2*af_flag + np.random.normal(0, 2, N)
).astype("float32")

# Assemble multi-input X representations
X_tab = np.stack([age, bmi, sbp], axis=1).astype("float32")
X_img = np.stack([ef, lvmass], axis=1).astype("float32")

X_single = np.concatenate([X_tab, X_img], axis=1)  # (N,5) single-array input
X_dict = {"clinical": X_tab, "imaging": X_img}     # dict input (multi-input)
X_list = [X_tab, X_img]                            # list input (multi-input)

# Multi-output y as dict (for multi-output model training)
y_multiout = {"mace": y_main.astype("int32"), "ef_drop": y_future_ef_drop.astype("float32")}

df_preview = pd.DataFrame({
    "patient_id": patient_ids,
    "age": age, "bmi": bmi, "sbp": sbp,
    "ef": ef, "lvmass": lvmass,
    "af": af_flag, "lvh": lvh_flag, "cond": cond_flag,
    "mace_1y": y_main,
    "future_ef_drop": y_future_ef_drop
})
display(df_preview.head(8))

print("Shapes:")
print("  X_single:", X_single.shape)
print("  X_tab   :", X_tab.shape, "X_img:", X_img.shape)
print("  y_main  :", y_main.shape, "y_multi:", y_multi.shape, "y_future:", y_future_ef_drop.shape)
print("  #patients:", len(np.unique(patient_ids)), "N:", len(patient_ids))
print("Label prevalence:")
print("  mace_1y mean:", y_main.mean().round(3))
print("  AF/LVH/COND means:", y_multi.mean(axis=0).round(3))



## Step 2 — Keras model factories (binary, multilabel, regression, multi-input, multi-output)

We’ll keep models small so CV runs quickly.

**Judgment numbers:** during training, you should see metrics improve slightly across epochs; we run only ~3 epochs for speed.


In [ ]:

def build_mace_mlp(input_dim=5):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

def build_multilabel_mlp(input_dim=5, C=3):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(32, activation="relu")(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(C, activation="sigmoid")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy")
    return model

def build_reg_mlp(input_dim=5):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(32, activation="relu")(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="linear")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model

def build_multiinput_dict_model():
    inp_clin = layers.Input(shape=(3,), name="clinical")
    inp_img  = layers.Input(shape=(2,), name="imaging")

    x1 = layers.Dense(16, activation="relu")(inp_clin)
    x2 = layers.Dense(16, activation="relu")(inp_img)
    x = layers.Concatenate()([x1, x2])
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model({"clinical": inp_clin, "imaging": inp_img}, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

def build_multiinput_list_model():
    # list inputs in the same order as X_list = [X_tab, X_img]
    inp_clin = layers.Input(shape=(3,))
    inp_img  = layers.Input(shape=(2,))

    x1 = layers.Dense(16, activation="relu")(inp_clin)
    x2 = layers.Dense(16, activation="relu")(inp_img)
    x = layers.Concatenate()([x1, x2])
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model([inp_clin, inp_img], out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

def build_multioutput_model():
    inp = layers.Input(shape=(X_single.shape[1],), name="features")
    x = layers.Dense(32, activation="relu")(inp)
    x = layers.Dropout(0.2)(x)

    mace = layers.Dense(1, activation="sigmoid", name="mace")(x)
    ef_drop = layers.Dense(1, activation="linear", name="ef_drop")(x)

    model = keras.Model(inp, {"mace": mace, "ef_drop": ef_drop})
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss={"mace": "binary_crossentropy", "ef_drop": "mse"},
        loss_weights={"mace": 1.0, "ef_drop": 0.3},
        metrics={"mace": ["accuracy"]},
    )
    return model



## Step 3 — TrustCVValidator: show leakage if you use the wrong CV method

### 3A) BAD configuration (expected leakage failure)
We use `method="stratified_kfold"` and pass `groups=patient_ids`.
In sklearn, `StratifiedKFold` ignores `groups`, so the same patient can appear in both train and test folds.

**Judgment numbers:**  
- Expect `Leakage Check: FAILED` (or a warning about group leakage).
- Metrics may look *slightly optimistic* vs. group-safe.

### 3B) GOOD configuration (patient-safe)
We use `method="stratified_grouped_kfold"` so splits are stratified **and** group-separated.

**Judgment numbers:**  
- Expect `Leakage Check: PASSED`.
- Performance may drop a bit (more realistic).


In [ ]:

# --- BAD: groups ignored -> leakage likely ---
bad = TrustCVValidator(
    method="stratified_kfold",
    n_splits=5,
    shuffle=True,
    random_state=SEED,
    check_leakage=True,
    check_balance=True,
)

res_bad = bad.validate(
    model=KerasSkWrap(
        build_fn=lambda input_shape=None, n_classes=None: build_mace_mlp(input_dim=X_single.shape[1]),
        epochs=3, batch_size=64, verbose=0,
        task="binary", threshold=0.5
    ),
    X=X_single,
    y=y_main,
    groups=patient_ids,
)

print(res_bad.summary())

# --- GOOD: stratified + group separated ---
good = TrustCVValidator(
    method="stratified_grouped_kfold",
    n_splits=5,
    shuffle=True,
    random_state=SEED,
    check_leakage=True,
    check_balance=True,
)

res_good = good.validate(
    model=KerasSkWrap(
        build_fn=lambda input_shape=None, n_classes=None: build_mace_mlp(input_dim=X_single.shape[1]),
        epochs=3, batch_size=64, verbose=0,
        task="binary", threshold=0.5
    ),
    X=X_single,
    y=y_main,
    groups=patient_ids,
)

print(res_good.summary())



## Step 4 — UniversalCVRunner: per-fold metrics + patient overlap check

This is the most educational output because you see fold-by-fold behavior.

We use **`StratifiedGroupKFold`** (patient-safe + label stratification).

**Judgment numbers:**  
- `patient_overlap` should be **0** in every fold.  
- If overlap > 0, you have leakage.  
- Compare fold metric variance (std) to understand stability.


In [ ]:

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
runner = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=1)

mace_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_mace_mlp(input_dim=X_single.shape[1]),
    epochs=3, batch_size=64, verbose=0,
    task="binary", threshold=0.5
)

results = runner.run(model=mace_clf, data=(X_single, y_main), groups=patient_ids)

fold_rows = []
for i, (tr_idx, te_idx) in enumerate(results.indices):
    y_true = y_main[te_idx]
    y_prob = results.probabilities[i]
    p1 = y_prob[:, 1] if y_prob.ndim == 2 else y_prob
    y_pred = (p1 >= 0.5).astype(int)

    overlap = len(set(patient_ids[tr_idx]).intersection(set(patient_ids[te_idx])))

    fold_rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "n_test": len(te_idx),
        "n_patients_test": len(np.unique(patient_ids[te_idx])),
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, p1),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    })

df_folds = pd.DataFrame(fold_rows)
display(df_folds)

print("Mean +/- std")
print(df_folds[["accuracy","roc_auc","precision","recall","f1"]].agg(["mean","std"]))



## Step 5 — Multi-input model (dict input) with UniversalCVRunner + KerasSkWrap

**Scenario:** A typical medical pipeline has multiple modalities (clinical + imaging).  
We model them as **two branches** merged later.

We pass `X_dict` directly (no helper class):  
```python
X_dict = {"clinical": X_tab, "imaging": X_img}
```

**Judgment numbers:**  
- Patient overlap should stay 0  
- AUC should be similar to the single-array model (it’s the same info, just structured)


In [ ]:

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
runner_mi = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=1)

mi_dict_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multiinput_dict_model(),
    epochs=3, batch_size=64, verbose=0,
    task="binary", threshold=0.5
)

mi_dict_results = runner_mi.run(model=mi_dict_clf, data=(X_dict, y_main), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(mi_dict_results.indices):
    y_true = y_main[te]
    p1 = mi_dict_results.probabilities[i][:, 1]
    y_pred = (p1 >= 0.5).astype(int)
    overlap = len(set(patient_ids[tr]).intersection(set(patient_ids[te])))
    rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, p1),
    })
df_mi_dict = pd.DataFrame(rows)
display(df_mi_dict)
print(df_mi_dict[["accuracy","roc_auc"]].agg(["mean","std"]))



## Step 6 — Multi-input model (list input)

Same idea as Step 5, but your data is a list:
```python
X_list = [X_tab, X_img]
```

**Judgment numbers:** results should be close to the dict-input model.


In [ ]:

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
runner_mi_list = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=1)

mi_list_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multiinput_list_model(),
    epochs=3, batch_size=64, verbose=0,
    task="binary", threshold=0.5
)

mi_list_results = runner_mi_list.run(model=mi_list_clf, data=(X_list, y_main), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(mi_list_results.indices):
    y_true = y_main[te]
    p1 = mi_list_results.probabilities[i][:, 1]
    y_pred = (p1 >= 0.5).astype(int)
    overlap = len(set(patient_ids[tr]).intersection(set(patient_ids[te])))
    rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, p1),
    })
df_mi_list = pd.DataFrame(rows)
display(df_mi_list)
print(df_mi_list[["accuracy","roc_auc"]].agg(["mean","std"]))



## Step 7 — Multi-label classification (sigmoid C=3 outputs)

**Scenario:** Predict multiple ECG/echo flags at once (AF/LVH/Conduction).  
Output is `(N, 3)` with sigmoid activations.

We use **GroupKFold** (patient-safe). True multilabel stratification is a separate topic; here we focus on showing TrustCV’s ability to run multilabel DL in CV.

**Judgment numbers:**  
- Compare **micro-F1** vs **macro-F1**  
  - micro-F1 weights frequent labels more  
  - macro-F1 treats each label equally, so it’s more sensitive to rare labels
- Check per-fold label prevalence (columns `prev_AF`, `prev_LVH`, `prev_COND`)


In [ ]:

cv_g = GroupKFold(n_splits=3)
runner_ml = UniversalCVRunner(cv_splitter=cv_g, framework="sklearn", verbose=1)

ml_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multilabel_mlp(input_dim=X_single.shape[1], C=3),
    epochs=3, batch_size=64, verbose=0,
    task="multilabel", threshold=0.5
)

ml_results = runner_ml.run(model=ml_clf, data=(X_single, y_multi), groups=patient_ids)

rows = []
for i, (tr, te) in enumerate(ml_results.indices):
    y_true = y_multi[te]
    y_pred = ml_results.predictions[i]  # expected (n,3) binary
    overlap = len(set(patient_ids[tr]).intersection(set(patient_ids[te])))

    rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "prev_AF": y_true[:,0].mean(),
        "prev_LVH": y_true[:,1].mean(),
        "prev_COND": y_true[:,2].mean(),
    })
df_ml = pd.DataFrame(rows)
display(df_ml)
print(df_ml[["micro_f1","macro_f1"]].agg(["mean","std"]))



## Step 8 — Regression (continuous y)

**Scenario:** Predict future EF drop as a continuous outcome.

**Judgment numbers:**  
- RMSE is in “EF drop units” (lower is better)  
- R² near 0 means weak prediction; >0 indicates some signal captured


In [ ]:

cv_g = GroupKFold(n_splits=3)
runner_reg = UniversalCVRunner(cv_splitter=cv_g, framework="sklearn", verbose=1)

reg_model = KerasRegressorWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_reg_mlp(input_dim=X_single.shape[1]),
    epochs=3, batch_size=64, verbose=0
)

reg_results = runner_reg.run(model=reg_model, data=(X_single, y_future_ef_drop), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(reg_results.indices):
    y_true = y_future_ef_drop[te]
    y_hat = np.asarray(reg_results.predictions[i]).reshape(-1)
    overlap = len(set(patient_ids[tr]).intersection(set(patient_ids[te])))

    rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "rmse": mean_squared_error(y_true, y_hat, squared=False),
        "r2": r2_score(y_true, y_hat),
    })
df_reg = pd.DataFrame(rows)
display(df_reg)
print(df_reg[["rmse","r2"]].agg(["mean","std"]))



## Step 9 — Multi-output model (classification + regression heads)

**Scenario:** A single network predicts:
- MACE risk (classification head `mace`)
- EF drop (regression head `ef_drop`)

We pass **dict y** directly:
```python
y_multiout = {"mace": y_main, "ef_drop": y_future_ef_drop}
```

Then we evaluate each head by selecting it:
- `KerasSkWrap(..., output_key="mace")`
- `KerasRegressorWrap(..., output_key="ef_drop")`

**Judgment numbers:**  
- Compare multi-output classification metrics vs the single-task classifier (should be similar)
- Regression head metrics might change slightly due to multi-task coupling


In [ ]:

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
runner_mo = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=1)

# --- classification head ---
mo_clf = KerasSkWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multioutput_model(),
    epochs=3, batch_size=64, verbose=0,
    task="binary", threshold=0.5,
    output_key="mace"
)
mo_clf_results = runner_mo.run(model=mo_clf, data=(X_single, y_multiout), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(mo_clf_results.indices):
    y_true = y_main[te]
    p1 = mo_clf_results.probabilities[i][:, 1]
    y_pred = (p1 >= 0.5).astype(int)
    overlap = len(set(patient_ids[tr]).intersection(set(patient_ids[te])))
    rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, p1),
    })
df_mo_clf = pd.DataFrame(rows)
display(df_mo_clf)
print(df_mo_clf[["accuracy","roc_auc"]].agg(["mean","std"]))

# --- regression head ---
mo_reg = KerasRegressorWrap(
    build_fn=lambda input_shape=None, n_classes=None: build_multioutput_model(),
    epochs=3, batch_size=64, verbose=0,
    output_key="ef_drop"
)
mo_reg_results = runner_mo.run(model=mo_reg, data=(X_single, y_multiout), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(mo_reg_results.indices):
    y_true = y_future_ef_drop[te]
    y_hat = np.asarray(mo_reg_results.predictions[i]).reshape(-1)
    overlap = len(set(patient_ids[tr]).intersection(set(patient_ids[te])))
    rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "rmse": mean_squared_error(y_true, y_hat, squared=False),
        "r2": r2_score(y_true, y_hat),
    })
df_mo_reg = pd.DataFrame(rows)
display(df_mo_reg)
print(df_mo_reg[["rmse","r2"]].agg(["mean","std"]))



## Step 10 — Custom training loop (`train_step`) *and* tf.data.Dataset (no IndexableTFDataset)

To keep the notebook free from `IndexableTFDataset`, we do dataset conversion **inside the estimator**.

**What we demonstrate:**
- a Keras `Model` subclass implementing a custom `train_step`
- training uses `tf.data.Dataset.from_tensor_slices(...).batch(...).prefetch(...)` each fold
- CV remains automatic (Runner splits arrays; the estimator converts fold arrays → dataset)

**Judgment numbers:**
- metrics should be close to the standard binary classifier but may differ slightly
- patient overlap should still be 0 with StratifiedGroupKFold


In [ ]:

class CustomTrainStepModel(keras.Model):
    def __init__(self, input_dim):
        super().__init__()
        self.d1 = layers.Dense(32, activation="relu")
        self.d2 = layers.Dense(16, activation="relu")
        self.out = layers.Dense(1, activation="sigmoid")

        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc = keras.metrics.BinaryAccuracy(name="acc")

    def call(self, x, training=False):
        x = self.d1(x)
        x = self.d2(x)
        return self.out(x)

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc]

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compiled_loss(y, y_pred, regularization_losses=self.losses)

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.loss_tracker.update_state(loss)
        self.acc.update_state(y, y_pred)
        return {"loss": self.loss_tracker.result(), "acc": self.acc.result()}

def build_custom_model():
    m = CustomTrainStepModel(input_dim=X_single.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy")
    return m

# Small subclass: convert fold arrays -> tf.data.Dataset inside fit
class KerasSkWrapAsDataset(KerasSkWrap):
    def fit(self, X, y=None, **fit_kwargs):
        # X and y arrive as fold arrays from the runner; convert to dataset.
        ds = tf.data.Dataset.from_tensor_slices((X, y)).batch(self.batch_size).prefetch(tf.data.AUTOTUNE)
        return super().fit(ds, y=None, **fit_kwargs)

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
runner_ds = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=1)

ds_clf = KerasSkWrapAsDataset(
    build_fn=lambda input_shape=None, n_classes=None: build_custom_model(),
    epochs=3, batch_size=64, verbose=0,
    task="binary", threshold=0.5,
    allow_dataset=True
)

ds_results = runner_ds.run(model=ds_clf, data=(X_single, y_main), groups=patient_ids)

rows=[]
for i, (tr, te) in enumerate(ds_results.indices):
    y_true = y_main[te]
    p1 = ds_results.probabilities[i][:, 1]
    y_pred = (p1 >= 0.5).astype(int)
    overlap = len(set(patient_ids[tr]).intersection(set(patient_ids[te])))
    rows.append({
        "fold": i+1,
        "patient_overlap": overlap,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, p1),
    })
df_ds = pd.DataFrame(rows)
display(df_ds)
print(df_ds[["accuracy","roc_auc"]].agg(["mean","std"]))



# Summary: what you proved with this notebook

You can now show (to users/reviewers/colleagues) that TrustCV supports deep learning CV end-to-end:

- **Group leakage detection** with `TrustCVValidator` (bad vs good splitters)
- **Per-fold reporting** with `UniversalCVRunner`
- **Multi-input** (`dict` or `list`) without extra wrappers
- **Multi-label** (sigmoid C outputs)
- **Regression**
- **Multi-output** with `output_key` selection
- **Custom `train_step`** with dataset training while keeping CV automatic

If you want, I can also generate a **mini unit-test notebook cell** (pytest-style in notebook) that asserts:
- patient_overlap==0 for group-safe CV
- predictions shapes are correct for binary/multilabel/regression/multioutput
